In [ ]:
# Вспомогательные библиотеки и типизация
import os
from tqdm import tqdm
import warnings
import zipfile
from pathlib import Path
from datasets import load_dataset
from typing import List, Tuple, Set, Dict, Union, Callable, Any

# Для работы с данными
import pandas as pd
import numpy as np

# Для визуализации
import matplotlib.pyplot as plt
import seaborn as sns

# Интерпретация
import shap 

# Библиотека для сохранения модели
import joblib

# Ансамбли
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

# Метрики и модели
from sklearn.base import clone
from sklearn.calibration import calibration_curve
from sklearn.model_selection import StratifiedKFold, KFold, learning_curve, train_test_split
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix, classification_report, precision_recall_curve, average_precision_score
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Для автоматического подбора параметров моделей
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from copy import deepcopy

# Настройка параметров библиотек
warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 6)

# Для полного вывода без сокращения
# pd.set_option('display.max_columns', None)
# pd.set_option('display.max_rows', None)

RANDOM_STATE: int = 42
ROOT: Path = Path.cwd()
DATA_DIR: Path = ROOT / "data" if (ROOT / "data").exists() else ROOT

print("✅ Окружение настроено")


✅ Окружение настроено


In [2]:
print(f"Root: {ROOT}")

Root: /home/yamshchikov/ML_practice/Torchvision_core_project/MMAD


In [4]:
df_train: pd.DataFrame = pd.read_csv('train.csv')
df_train

,AAGE,ACLSWKR,ADTIND,ADTOCC,AHGA,AHRSPAY,AHSCOL,AMARITL,AMJIND,AMJOCC,...,PEFNTVTY,PEMNTVTY,PENATVTY,PRCITSHP,SEOTR,TAXINC,VETQVA,VETYN,WKSWORK,CLASS
0,43,LocalGovernment,43,10,Mastersdegree(MAMSMEngMEdMSWMBA),0,Not_in_universe,Married-civilianspousepresent,Education,Professionalspecialty,...,United-States,United-States,United-States,Native-BornintheUnitedStates,0,Not_in_universe,2,52,94,50000+
1,0,Not_in_universe,0,0,Children,0,Not_in_universe,Nevermarried,Not_in_universeorchildren,Not_in_universe,...,United-States,United-States,United-States,Native-BornintheUnitedStates,0,Not_in_universe,0,0,94,-50000
2,44,Private,11,22,Somecollegebutnodegree,0,Not_in_universe,Married-civilianspousepresent,Manufacturing-durablegoods,Admsupportincludingclerical,...,United-States,United-States,United-States,Native-BornintheUnitedStates,0,Not_in_universe,2,52,94,50000+
3,43,Self-employed-incorporated,35,17,Bachelorsdegree(BAABBS),0,Not_in_universe,Married-civilianspousepresent,Financeinsuranceandrealestate,Sales,...,United-States,United-States,United-States,Native-BornintheUnitedStates,0,Not_in_universe,2,52,94,50000+
4,39,Private,24,26,Bachelorsdegree(BAABBS),0,Not_in_universe,Married-civilianspousepresent,Manufacturing-nondurablegoods,Admsupportincludingclerical,...,United-States,United-States,United-States,Native-BornintheUnitedStates,0,Not_in_universe,2,52,94,50000+
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,57,Private,42,26,Somecollegebutnodegree,0,Not_in_universe,Married-civilianspousepresent,Medicalexcepthospital,Admsupportincludingclerical,...,United-States,El-Salvador,United-States,Native-BornintheUnitedStates,0,Not_in_universe,2,52,94,50000+
9996,13,Not_in_universe,0,0,Children,0,Not_in_universe,Nevermarried,Not_in_universeorchildren,Not_in_universe,...,United-States,United-States,United-States,Native-BornintheUnitedStates,0,Not_in_universe,0,0,94,-50000
9997,5,Not_in_universe,0,0,Children,0,Not_in_universe,Nevermarried,Not_in_universeorchildren,Not_in_universe,...,United-States,United-States,United-States,Native-BornintheUnitedStates,0,Not_in_universe,0,0,94,-50000
9998,64,Not_in_universe,0,0,10thgrade,0,Not_in_universe,Married-civilianspousepresent,Not_in_universeorchildren,Not_in_universe,...,United-States,United-States,United-States,Native-BornintheUnitedStates,0,Not_in_universe,2,0,94,-50000


In [5]:
df_train.shape

(10000, 42)

In [13]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 42 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   AAGE      10000 non-null  int64  
 1   ACLSWKR   10000 non-null  object 
 2   ADTIND    10000 non-null  int64  
 3   ADTOCC    10000 non-null  int64  
 4   AHGA      10000 non-null  object 
 5   AHRSPAY   10000 non-null  int64  
 6   AHSCOL    10000 non-null  object 
 7   AMARITL   10000 non-null  object 
 8   AMJIND    10000 non-null  object 
 9   AMJOCC    10000 non-null  object 
 10  ARACE     10000 non-null  object 
 11  AREORGN   10000 non-null  object 
 12  ASEX      10000 non-null  object 
 13  AUNMEM    10000 non-null  object 
 14  AUNTYPE   10000 non-null  object 
 15  AWKSTAT   10000 non-null  object 
 16  CAPGAIN   10000 non-null  int64  
 17  CAPLOSS   10000 non-null  int64  
 18  DIVVAL    10000 non-null  int64  
 19  FILESTAT  10000 non-null  object 
 20  GRINREG   10000 non-null  obj

In [6]:
df_train.quantile

<bound method DataFrame.quantile of       AAGE                     ACLSWKR  ADTIND  ADTOCC  \
0       43             LocalGovernment      43      10   
1        0             Not_in_universe       0       0   
2       44                     Private      11      22   
3       43  Self-employed-incorporated      35      17   
4       39                     Private      24      26   
...    ...                         ...     ...     ...   
9995    57                     Private      42      26   
9996    13             Not_in_universe       0       0   
9997     5             Not_in_universe       0       0   
9998    64             Not_in_universe       0       0   
9999    29             LocalGovernment      45      15   

                                  AHGA  AHRSPAY           AHSCOL  \
0     Mastersdegree(MAMSMEngMEdMSWMBA)        0  Not_in_universe   
1                             Children        0  Not_in_universe   
2               Somecollegebutnodegree        0  Not_in_univers

In [7]:
df_train.columns

Index(['AAGE', 'ACLSWKR', 'ADTIND', 'ADTOCC', 'AHGA', 'AHRSPAY', 'AHSCOL',
       'AMARITL', 'AMJIND', 'AMJOCC', 'ARACE', 'AREORGN', 'ASEX', 'AUNMEM',
       'AUNTYPE', 'AWKSTAT', 'CAPGAIN', 'CAPLOSS', 'DIVVAL', 'FILESTAT',
       'GRINREG', 'GRINST', 'HHDFMX', 'HHDREL', 'MARSUPWT', 'MIGMTR1',
       'MIGMTR3', 'MIGMTR4', 'MIGSAME', 'MIGSUN', 'NOEMP', 'PARENT',
       'PEFNTVTY', 'PEMNTVTY', 'PENATVTY', 'PRCITSHP', 'SEOTR', 'TAXINC',
       'VETQVA', 'VETYN', 'WKSWORK', 'CLASS'],
      dtype='object')

In [8]:
df_train.dtypes

AAGE          int64
ACLSWKR      object
ADTIND        int64
ADTOCC        int64
AHGA         object
AHRSPAY       int64
AHSCOL       object
AMARITL      object
AMJIND       object
AMJOCC       object
ARACE        object
AREORGN      object
ASEX         object
AUNMEM       object
AUNTYPE      object
AWKSTAT      object
CAPGAIN       int64
CAPLOSS       int64
DIVVAL        int64
FILESTAT     object
GRINREG      object
GRINST       object
HHDFMX       object
HHDREL       object
MARSUPWT    float64
MIGMTR1      object
MIGMTR3      object
MIGMTR4      object
MIGSAME      object
MIGSUN       object
NOEMP         int64
PARENT       object
PEFNTVTY     object
PEMNTVTY     object
PENATVTY     object
PRCITSHP     object
SEOTR         int64
TAXINC       object
VETQVA        int64
VETYN         int64
WKSWORK       int64
CLASS        object
dtype: object

In [9]:
df_train.index

RangeIndex(start=0, stop=10000, step=1)

In [10]:
df_train.axes

[RangeIndex(start=0, stop=10000, step=1),
 Index(['AAGE', 'ACLSWKR', 'ADTIND', 'ADTOCC', 'AHGA', 'AHRSPAY', 'AHSCOL',
        'AMARITL', 'AMJIND', 'AMJOCC', 'ARACE', 'AREORGN', 'ASEX', 'AUNMEM',
        'AUNTYPE', 'AWKSTAT', 'CAPGAIN', 'CAPLOSS', 'DIVVAL', 'FILESTAT',
        'GRINREG', 'GRINST', 'HHDFMX', 'HHDREL', 'MARSUPWT', 'MIGMTR1',
        'MIGMTR3', 'MIGMTR4', 'MIGSAME', 'MIGSUN', 'NOEMP', 'PARENT',
        'PEFNTVTY', 'PEMNTVTY', 'PENATVTY', 'PRCITSHP', 'SEOTR', 'TAXINC',
        'VETQVA', 'VETYN', 'WKSWORK', 'CLASS'],
       dtype='object')]

In [11]:
df_train[['VETYN','CLASS']]

,VETYN,CLASS
0,52,50000+
1,0,-50000
2,52,50000+
3,52,50000+
4,52,50000+
...,...,...
9995,52,50000+
9996,0,-50000
9997,0,-50000
9998,0,-50000


In [14]:
print("Типы данных train:\n", df_train.dtypes)
print("Основные статистики для числовых признаков:\n", df_train.describe())

cat_cols_train = df_train.select_dtypes(include=['object']).columns
if len(cat_cols_train) > 0:
    print("Основные статистики для категориальных признаков:\n", df_train[cat_cols_train].describe())

Типы данных train:
 AAGE          int64
ACLSWKR      object
ADTIND        int64
ADTOCC        int64
AHGA         object
AHRSPAY       int64
AHSCOL       object
AMARITL      object
AMJIND       object
AMJOCC       object
ARACE        object
AREORGN      object
ASEX         object
AUNMEM       object
AUNTYPE      object
AWKSTAT      object
CAPGAIN       int64
CAPLOSS       int64
DIVVAL        int64
FILESTAT     object
GRINREG      object
GRINST       object
HHDFMX       object
HHDREL       object
MARSUPWT    float64
MIGMTR1      object
MIGMTR3      object
MIGMTR4      object
MIGSAME      object
MIGSUN       object
NOEMP         int64
PARENT       object
PEFNTVTY     object
PEMNTVTY     object
PENATVTY     object
PRCITSHP     object
SEOTR         int64
TAXINC       object
VETQVA        int64
VETYN         int64
WKSWORK       int64
CLASS        object
dtype: object
Основные статистики для числовых признаков:
                AAGE        ADTIND        ADTOCC       AHRSPAY       CAPGAIN  \
co

In [15]:
duplicates_train = df_train.duplicated(subset=['CLASS']).sum()
print(f"Количество дубликатов по 'CLASS' в train: {duplicates_train}")

Количество дубликатов по 'CLASS' в train: 9998


In [16]:
print(f"\nКолонки в train: {df_train.columns.tolist()}")
print(f"\nТипы данных:\n{df_train.dtypes.value_counts()}")


Колонки в train: ['AAGE', 'ACLSWKR', 'ADTIND', 'ADTOCC', 'AHGA', 'AHRSPAY', 'AHSCOL', 'AMARITL', 'AMJIND', 'AMJOCC', 'ARACE', 'AREORGN', 'ASEX', 'AUNMEM', 'AUNTYPE', 'AWKSTAT', 'CAPGAIN', 'CAPLOSS', 'DIVVAL', 'FILESTAT', 'GRINREG', 'GRINST', 'HHDFMX', 'HHDREL', 'MARSUPWT', 'MIGMTR1', 'MIGMTR3', 'MIGMTR4', 'MIGSAME', 'MIGSUN', 'NOEMP', 'PARENT', 'PEFNTVTY', 'PEMNTVTY', 'PENATVTY', 'PRCITSHP', 'SEOTR', 'TAXINC', 'VETQVA', 'VETYN', 'WKSWORK', 'CLASS']

Типы данных:
object     29
int64      12
float64     1
Name: count, dtype: int64


In [17]:
num_cols_train: pd.Index = df_train.select_dtypes(include=["int64", "float64"]).columns
cat_cols_train: pd.Index = df_train.select_dtypes(include=["object"]).columns

print(f"Числовые: {len(num_cols_train)}")
print(f"Категориальные: {len(cat_cols_train)}")

Числовые: 13
Категориальные: 29


In [18]:
float_train_cols: List[str] = df_train.select_dtypes(include=['float64']).columns.tolist()
float_train_cols

['MARSUPWT']

In [19]:
int_train_cols: List[str] = df_train.select_dtypes(include=['int64']).columns.tolist()
int_train_cols

['AAGE',
 'ADTIND',
 'ADTOCC',
 'AHRSPAY',
 'CAPGAIN',
 'CAPLOSS',
 'DIVVAL',
 'NOEMP',
 'SEOTR',
 'VETQVA',
 'VETYN',
 'WKSWORK']

In [20]:
object_train_cols: List[str] = df_train.select_dtypes(include=['object']).columns.tolist()
object_train_cols

['ACLSWKR',
 'AHGA',
 'AHSCOL',
 'AMARITL',
 'AMJIND',
 'AMJOCC',
 'ARACE',
 'AREORGN',
 'ASEX',
 'AUNMEM',
 'AUNTYPE',
 'AWKSTAT',
 'FILESTAT',
 'GRINREG',
 'GRINST',
 'HHDFMX',
 'HHDREL',
 'MIGMTR1',
 'MIGMTR3',
 'MIGMTR4',
 'MIGSAME',
 'MIGSUN',
 'PARENT',
 'PEFNTVTY',
 'PEMNTVTY',
 'PENATVTY',
 'PRCITSHP',
 'TAXINC',
 'CLASS']

In [29]:
df_train[(df_train['AHGA']=='Mastersdegree(MAMSMEngMEdMSWMBA)')&(df_train['AAGE']>30)].shape[0]/df_train[df_train['AAGE']>30].shape[0]

0.1210366694237662

In [30]:
df_train.iloc[4]
df_train['AAGE']
df_train[:3]

,AAGE,ACLSWKR,ADTIND,ADTOCC,AHGA,AHRSPAY,AHSCOL,AMARITL,AMJIND,AMJOCC,...,PEFNTVTY,PEMNTVTY,PENATVTY,PRCITSHP,SEOTR,TAXINC,VETQVA,VETYN,WKSWORK,CLASS
0,43,LocalGovernment,43,10,Mastersdegree(MAMSMEngMEdMSWMBA),0,Not_in_universe,Married-civilianspousepresent,Education,Professionalspecialty,...,United-States,United-States,United-States,Native-BornintheUnitedStates,0,Not_in_universe,2,52,94,50000+
1,0,Not_in_universe,0,0,Children,0,Not_in_universe,Nevermarried,Not_in_universeorchildren,Not_in_universe,...,United-States,United-States,United-States,Native-BornintheUnitedStates,0,Not_in_universe,0,0,94,-50000
2,44,Private,11,22,Somecollegebutnodegree,0,Not_in_universe,Married-civilianspousepresent,Manufacturing-durablegoods,Admsupportincludingclerical,...,United-States,United-States,United-States,Native-BornintheUnitedStates,0,Not_in_universe,2,52,94,50000+


In [31]:
df_train[df_train['AAGE']>30]

,AAGE,ACLSWKR,ADTIND,ADTOCC,AHGA,AHRSPAY,AHSCOL,AMARITL,AMJIND,AMJOCC,...,PEFNTVTY,PEMNTVTY,PENATVTY,PRCITSHP,SEOTR,TAXINC,VETQVA,VETYN,WKSWORK,CLASS
0,43,LocalGovernment,43,10,Mastersdegree(MAMSMEngMEdMSWMBA),0,Not_in_universe,Married-civilianspousepresent,Education,Professionalspecialty,...,United-States,United-States,United-States,Native-BornintheUnitedStates,0,Not_in_universe,2,52,94,50000+
2,44,Private,11,22,Somecollegebutnodegree,0,Not_in_universe,Married-civilianspousepresent,Manufacturing-durablegoods,Admsupportincludingclerical,...,United-States,United-States,United-States,Native-BornintheUnitedStates,0,Not_in_universe,2,52,94,50000+
3,43,Self-employed-incorporated,35,17,Bachelorsdegree(BAABBS),0,Not_in_universe,Married-civilianspousepresent,Financeinsuranceandrealestate,Sales,...,United-States,United-States,United-States,Native-BornintheUnitedStates,0,Not_in_universe,2,52,94,50000+
4,39,Private,24,26,Bachelorsdegree(BAABBS),0,Not_in_universe,Married-civilianspousepresent,Manufacturing-nondurablegoods,Admsupportincludingclerical,...,United-States,United-States,United-States,Native-BornintheUnitedStates,0,Not_in_universe,2,52,94,50000+
6,70,Not_in_universe,0,0,Highschoolgraduate,0,Not_in_universe,Married-civilianspousepresent,Not_in_universeorchildren,Not_in_universe,...,United-States,United-States,United-States,Native-BornintheUnitedStates,0,Not_in_universe,2,0,94,-50000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9992,90,Not_in_universe,0,0,Highschoolgraduate,0,Not_in_universe,Married-civilianspousepresent,Not_in_universeorchildren,Not_in_universe,...,United-States,United-States,United-States,Native-BornintheUnitedStates,0,Not_in_universe,2,26,94,50000+
9993,72,Not_in_universe,0,0,Highschoolgraduate,0,Not_in_universe,Nevermarried,Not_in_universeorchildren,Not_in_universe,...,Mexico,Mexico,United-States,Native-BornintheUnitedStates,0,Not_in_universe,2,0,94,-50000
9994,53,Private,33,19,7thand8thgrade,475,Not_in_universe,Married-civilianspousepresent,Retailtrade,Sales,...,United-States,United-States,United-States,Native-BornintheUnitedStates,0,Not_in_universe,2,52,94,-50000
9995,57,Private,42,26,Somecollegebutnodegree,0,Not_in_universe,Married-civilianspousepresent,Medicalexcepthospital,Admsupportincludingclerical,...,United-States,El-Salvador,United-States,Native-BornintheUnitedStates,0,Not_in_universe,2,52,94,50000+


In [32]:
df_train['ACLSWKR'].value_counts()

ACLSWKR
Private                          4742
Not_in_universe                  2973
Self-employed-notIncorporated     699
LocalGovernment                   509
Self-employed-incorporated        491
StateGovernment                   298
FederalGovernment                 267
NeverWorked                        16
WithoutPay                          5
Name: count, dtype: int64

In [33]:
for name in df_train.columns:
    if df_train[name].dtype=='object':
        s=df_train[name].value_counts()
        if s[0]>int(0.8*df_train.index.size):
            print(name,' poor informativity ', s)
        if s[-1]<10:
            print(name,' small grads', s)
    else:
        if df_train[name].var()==0:
            print(name,'feature is constant')
    

ACLSWKR  small grads ACLSWKR
Private                          4742
Not_in_universe                  2973
Self-employed-notIncorporated     699
LocalGovernment                   509
Self-employed-incorporated        491
StateGovernment                   298
FederalGovernment                 267
NeverWorked                        16
WithoutPay                          5
Name: count, dtype: int64
AHSCOL  poor informativity  AHSCOL
Not_in_universe        9650
Highschool              176
Collegeoruniversity     174
Name: count, dtype: int64
AMJIND  small grads AMJIND
Not_in_universeorchildren          2989
Manufacturing-durablegoods          861
Retailtrade                         725
Financeinsuranceandrealestate       608
Education                           543
Manufacturing-nondurablegoods       513
Otherprofessionalservices           498
Businessandrepairservices           403
Construction                        379
Publicadministration                379
Medicalexcepthospital          

/tmp/ipykernel_1426323/3478013104.py:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if s[0]>int(0.8*df_train.index.size):
/tmp/ipykernel_1426323/3478013104.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if s[-1]<10:
/tmp/ipykernel_1426323/3478013104.py:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if s[0]>int(0.8*df_train.index.size):
/tmp/ipykernel_1426323/3478013104.py:6: FutureWarning: Series.__getitem__ treating keys a

In [34]:
df_train.replace(to_replace={'AMARITL': {'Separated': 'Otherwise','Widowed': 'Otherwise','Divorsed':'Otherwise'}})

,AAGE,ACLSWKR,ADTIND,ADTOCC,AHGA,AHRSPAY,AHSCOL,AMARITL,AMJIND,AMJOCC,...,PEFNTVTY,PEMNTVTY,PENATVTY,PRCITSHP,SEOTR,TAXINC,VETQVA,VETYN,WKSWORK,CLASS
0,43,LocalGovernment,43,10,Mastersdegree(MAMSMEngMEdMSWMBA),0,Not_in_universe,Married-civilianspousepresent,Education,Professionalspecialty,...,United-States,United-States,United-States,Native-BornintheUnitedStates,0,Not_in_universe,2,52,94,50000+
1,0,Not_in_universe,0,0,Children,0,Not_in_universe,Nevermarried,Not_in_universeorchildren,Not_in_universe,...,United-States,United-States,United-States,Native-BornintheUnitedStates,0,Not_in_universe,0,0,94,-50000
2,44,Private,11,22,Somecollegebutnodegree,0,Not_in_universe,Married-civilianspousepresent,Manufacturing-durablegoods,Admsupportincludingclerical,...,United-States,United-States,United-States,Native-BornintheUnitedStates,0,Not_in_universe,2,52,94,50000+
3,43,Self-employed-incorporated,35,17,Bachelorsdegree(BAABBS),0,Not_in_universe,Married-civilianspousepresent,Financeinsuranceandrealestate,Sales,...,United-States,United-States,United-States,Native-BornintheUnitedStates,0,Not_in_universe,2,52,94,50000+
4,39,Private,24,26,Bachelorsdegree(BAABBS),0,Not_in_universe,Married-civilianspousepresent,Manufacturing-nondurablegoods,Admsupportincludingclerical,...,United-States,United-States,United-States,Native-BornintheUnitedStates,0,Not_in_universe,2,52,94,50000+
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,57,Private,42,26,Somecollegebutnodegree,0,Not_in_universe,Married-civilianspousepresent,Medicalexcepthospital,Admsupportincludingclerical,...,United-States,El-Salvador,United-States,Native-BornintheUnitedStates,0,Not_in_universe,2,52,94,50000+
9996,13,Not_in_universe,0,0,Children,0,Not_in_universe,Nevermarried,Not_in_universeorchildren,Not_in_universe,...,United-States,United-States,United-States,Native-BornintheUnitedStates,0,Not_in_universe,0,0,94,-50000
9997,5,Not_in_universe,0,0,Children,0,Not_in_universe,Nevermarried,Not_in_universeorchildren,Not_in_universe,...,United-States,United-States,United-States,Native-BornintheUnitedStates,0,Not_in_universe,0,0,94,-50000
9998,64,Not_in_universe,0,0,10thgrade,0,Not_in_universe,Married-civilianspousepresent,Not_in_universeorchildren,Not_in_universe,...,United-States,United-States,United-States,Native-BornintheUnitedStates,0,Not_in_universe,2,0,94,-50000


In [35]:
df_train.loc[df_train['AMARITL']!='Married-civilianspousepresent','AMARITL']='Otherwise'
df_train

,AAGE,ACLSWKR,ADTIND,ADTOCC,AHGA,AHRSPAY,AHSCOL,AMARITL,AMJIND,AMJOCC,...,PEFNTVTY,PEMNTVTY,PENATVTY,PRCITSHP,SEOTR,TAXINC,VETQVA,VETYN,WKSWORK,CLASS
0,43,LocalGovernment,43,10,Mastersdegree(MAMSMEngMEdMSWMBA),0,Not_in_universe,Married-civilianspousepresent,Education,Professionalspecialty,...,United-States,United-States,United-States,Native-BornintheUnitedStates,0,Not_in_universe,2,52,94,50000+
1,0,Not_in_universe,0,0,Children,0,Not_in_universe,Otherwise,Not_in_universeorchildren,Not_in_universe,...,United-States,United-States,United-States,Native-BornintheUnitedStates,0,Not_in_universe,0,0,94,-50000
2,44,Private,11,22,Somecollegebutnodegree,0,Not_in_universe,Married-civilianspousepresent,Manufacturing-durablegoods,Admsupportincludingclerical,...,United-States,United-States,United-States,Native-BornintheUnitedStates,0,Not_in_universe,2,52,94,50000+
3,43,Self-employed-incorporated,35,17,Bachelorsdegree(BAABBS),0,Not_in_universe,Married-civilianspousepresent,Financeinsuranceandrealestate,Sales,...,United-States,United-States,United-States,Native-BornintheUnitedStates,0,Not_in_universe,2,52,94,50000+
4,39,Private,24,26,Bachelorsdegree(BAABBS),0,Not_in_universe,Married-civilianspousepresent,Manufacturing-nondurablegoods,Admsupportincludingclerical,...,United-States,United-States,United-States,Native-BornintheUnitedStates,0,Not_in_universe,2,52,94,50000+
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,57,Private,42,26,Somecollegebutnodegree,0,Not_in_universe,Married-civilianspousepresent,Medicalexcepthospital,Admsupportincludingclerical,...,United-States,El-Salvador,United-States,Native-BornintheUnitedStates,0,Not_in_universe,2,52,94,50000+
9996,13,Not_in_universe,0,0,Children,0,Not_in_universe,Otherwise,Not_in_universeorchildren,Not_in_universe,...,United-States,United-States,United-States,Native-BornintheUnitedStates,0,Not_in_universe,0,0,94,-50000
9997,5,Not_in_universe,0,0,Children,0,Not_in_universe,Otherwise,Not_in_universeorchildren,Not_in_universe,...,United-States,United-States,United-States,Native-BornintheUnitedStates,0,Not_in_universe,0,0,94,-50000
9998,64,Not_in_universe,0,0,10thgrade,0,Not_in_universe,Married-civilianspousepresent,Not_in_universeorchildren,Not_in_universe,...,United-States,United-States,United-States,Native-BornintheUnitedStates,0,Not_in_universe,2,0,94,-50000


In [36]:
# 1. Заменяем значения сразу на числа (1 и 0), а не на строки ('1' и '0')
df_train['CLASS'] = df_train['CLASS'].replace({'50000+': 1, '-50000': 0})

# 2. Извлекаем таргет с явным указанием типа и используем to_numpy()
Y = df_train['CLASS'].to_numpy(dtype=np.int64)

# 3. Удаляем колонку
del df_train['CLASS']

Y

/tmp/ipykernel_1426323/251902047.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_train['CLASS'] = df_train['CLASS'].replace({'50000+': 1, '-50000': 0})


array([1, 0, 1, ..., 0, 0, 1], shape=(10000,))

In [ ]:
def dummy_encode_categorical_columns(df, categorical_columns):
    result_df = deepcopy(df)
    for column in categorical_columns:
        result_df = pd.concat([result_df, pd.get_dummies(result_df[column], prefix=column, prefix_sep=': ')], axis=1)
        del result_df[column]
    return result_df

In [ ]:
categorical_columns: Set[str] = set(df_train.columns[df_train.dtypes == 'object'])
df_no_cat: pd.DataFrame = dummy_encode_categorical_columns(df_train, categorical_columns)
X_train: np.ndarray = df_no_cat.values.astype(np.float32)

In [39]:
df_no_cat

,AAGE,ADTIND,ADTOCC,AHRSPAY,CAPGAIN,CAPLOSS,DIVVAL,MARSUPWT,NOEMP,SEOTR,...,PRCITSHP: Native-BornintheUnitedStates,AREORGN: Allother,AREORGN: CentralorSouthAmerican,AREORGN: Chicano,AREORGN: Cuban,AREORGN: Donotknow,AREORGN: Mexican(Mexicano),AREORGN: Mexican-American,AREORGN: OtherSpanish,AREORGN: PuertoRican
0,43,43,10,0,0,0,2,1232.40,6,0,...,True,True,False,False,False,False,False,False,False,False
1,0,0,0,0,0,0,0,3062.92,0,0,...,True,True,False,False,False,False,False,False,False,False
2,44,11,22,0,0,0,1000,1502.40,6,0,...,True,True,False,False,False,False,False,False,False,False
3,43,35,17,0,99999,0,1000,295.30,2,0,...,True,True,False,False,False,False,False,False,False,False
4,39,24,26,0,0,0,0,2700.08,6,0,...,True,True,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,57,42,26,0,0,0,500,2115.52,1,0,...,True,True,False,False,False,False,False,False,False,False
9996,13,0,0,0,0,0,0,1732.77,0,0,...,True,True,False,False,False,False,False,False,False,False
9997,5,0,0,0,0,0,0,2326.47,0,0,...,True,True,False,False,False,False,False,False,False,False
9998,64,0,0,0,0,0,0,3217.88,0,0,...,True,True,False,False,False,False,False,False,False,False


In [40]:
X_train


array([[43., 43., 10., ...,  0.,  0.,  0.],
       [ 0.,  0.,  0., ...,  0.,  0.,  0.],
       [44., 11., 22., ...,  0.,  0.,  0.],
       ...,
       [ 5.,  0.,  0., ...,  0.,  0.,  0.],
       [64.,  0.,  0., ...,  0.,  0.,  0.],
       [29., 45., 15., ...,  0.,  0.,  0.]],
      shape=(10000, 367), dtype=float32)

In [41]:
cl=np.random.randint(2, size=10000)
ag=np.random.randint(low=31,high=100, size=10000)
ag

array([91, 84, 79, ..., 84, 88, 51], shape=(10000,))